# Tableau Data Preparation

This notebook creates the dashboard-ready session-level dataset for Tableau.

Input:

```text
Dataset/Processed Dataset/funnel_data.csv
```

Output:

```text
Dataset/Processed Dataset/tableau_session_funnel_data.csv
```

The base `funnel_data.csv` is created in the Funnel Analytics notebook from raw `events.csv`. This notebook adds reusable reporting and modeling features that are useful for Tableau dashboards and the baseline Logistic Regression notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load the validated session-level funnel dataset
from pathlib import Path

funnel_input_path = Path('../Dataset/Processed Dataset/funnel_data.csv')

if not funnel_input_path.exists():
    raise FileNotFoundError(
        "Input file not found: '../Dataset/Processed Dataset/funnel_data.csv'. "
        "Please run 'Notebooks/Funnel Analytics.ipynb' first to generate and save funnel_data.csv."
    )

funnel_df = pd.read_csv(funnel_input_path)

funnel_df.head()

In [ ]:
funnel_df.shape

## Create Dashboard-Ready Features

These features are not only model features. They are general session-level reporting features that can support Tableau dashboard views, segmentation, and purchase behavior analysis.

In [ ]:
# Create Tableau-ready session-level dataset
tableau_df = funnel_df.copy()

# Convert session_start to datetime for time-based reporting features.
tableau_df['session_start'] = pd.to_datetime(tableau_df['session_start'])

# Log-transform duration because session duration is highly skewed.
tableau_df['log_session_duration_mins'] = np.log1p(tableau_df['session_duration_mins'])

# Visitor frequency features.
visitor_session_counts = (
    tableau_df
    .groupby('visitorid')['session_key']
    .nunique()
    .reset_index(name='visitor_session_count')
)

tableau_df = tableau_df.merge(
    visitor_session_counts,
    on='visitorid',
    how='left'
)

tableau_df['is_repeat_visitor'] = (
    tableau_df['visitor_session_count'] > 1
).astype(int)

# Session start time features.
tableau_df['session_start_hour'] = tableau_df['session_start'].dt.hour
tableau_df['session_start_dayofweek'] = tableau_df['session_start'].dt.dayofweek
tableau_df['session_start_day_name'] = tableau_df['session_start'].dt.day_name()
tableau_df['is_weekend'] = tableau_df['session_start_dayofweek'].isin([5, 6]).astype(int)

tableau_df.head()

In [ ]:
# Validate the Tableau-ready dataset
validation_summary = pd.DataFrame({
    'check': [
        'Rows in base funnel dataset',
        'Rows in Tableau-ready dataset',
        'Unique session keys in Tableau-ready dataset',
        'Missing session keys',
        'Duplicate session keys'
    ],
    'value': [
        len(funnel_df),
        len(tableau_df),
        tableau_df['session_key'].nunique(),
        tableau_df['session_key'].isna().sum(),
        tableau_df.duplicated(subset=['session_key']).sum()
    ]
})

validation_summary

## Enhanced Feature Visual Checks

These checks inspect the reusable enhanced features before saving the Tableau-ready dataset.

### 1. Session Duration Before and After Log Transformation

Session duration is highly skewed. The log transformation reduces the effect of extreme long sessions, but it does not need to create a normal distribution.

In [ ]:
# Plot session duration before and after log transformation
# Use the 99th percentile cap for readability because duration is highly skewed.
duration_cap = tableau_df['session_duration_mins'].quantile(0.99)
log_duration_cap = tableau_df['log_session_duration_mins'].quantile(0.99)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(
    data=tableau_df[tableau_df['session_duration_mins'] <= duration_cap],
    x='session_duration_mins',
    bins=40,
    color='#4C78A8',
    ax=axes[0]
)
axes[0].set_title('Before Log Transformation')
axes[0].set_xlabel('Session duration (mins)')
axes[0].set_ylabel('Number of sessions')

sns.histplot(
    data=tableau_df[tableau_df['log_session_duration_mins'] <= log_duration_cap],
    x='log_session_duration_mins',
    bins=40,
    color='#54A24B',
    ax=axes[1]
)
axes[1].set_title('After Log Transformation')
axes[1].set_xlabel('log1p(session duration mins)')
axes[1].set_ylabel('Number of sessions')

plt.tight_layout()
plt.show()

### 2. Visitor Session Count Distribution

This shows how many visitors are one-time visitors versus visitors with multiple sessions.

In [ ]:
# Count plot for visitor_session_count
# Cap high repeat counts as 10+ so the chart remains readable.
visitor_count_plot = tableau_df[['visitorid', 'visitor_session_count']].drop_duplicates().copy()
visitor_count_plot['visitor_session_count_group'] = np.where(
    visitor_count_plot['visitor_session_count'] >= 10,
    '10+',
    visitor_count_plot['visitor_session_count'].astype(str)
)

order = [str(i) for i in range(1, 10)] + ['10+']
count_data = (
    visitor_count_plot['visitor_session_count_group']
    .value_counts()
    .reindex(order, fill_value=0)
    .reset_index()
)
count_data.columns = ['visitor_session_count_group', 'visitor_count']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(
    count_data['visitor_session_count_group'],
    count_data['visitor_count'],
    color='#4C78A8'
)

ax.set_title('Visitor Session Count Distribution')
ax.set_xlabel('Number of sessions per visitor')
ax.set_ylabel('Number of visitors')
ax.ticklabel_format(style='plain', axis='y')

for bar, count in zip(bars, count_data['visitor_count']):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f'{count:,.0f}',
        ha='center',
        va='bottom',
        fontsize=9
    )

plt.tight_layout()
plt.show()

### 3. One-Time vs Repeat Visitors

This chart summarizes the visitor population into one-time and repeat visitors.

In [ ]:
# Count plot for one-time vs repeat visitors
repeat_plot = (
    tableau_df[['visitorid', 'is_repeat_visitor']]
    .drop_duplicates()
    .copy()
)
repeat_plot['visitor_type'] = repeat_plot['is_repeat_visitor'].map({
    0: 'One-time visitor',
    1: 'Repeat visitor'
})

repeat_counts = (
    repeat_plot['visitor_type']
    .value_counts()
    .reindex(['One-time visitor', 'Repeat visitor'])
    .reset_index()
)
repeat_counts.columns = ['visitor_type', 'visitor_count']

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(
    repeat_counts['visitor_type'],
    repeat_counts['visitor_count'],
    color=['#F58518', '#54A24B']
)

ax.set_title('One-Time vs Repeat Visitors')
ax.set_xlabel('Visitor type')
ax.set_ylabel('Number of visitors')
ax.ticklabel_format(style='plain', axis='y')

for bar, count in zip(bars, repeat_counts['visitor_count']):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f'{count:,.0f}',
        ha='center',
        va='bottom',
        fontsize=10,
        fontweight='bold'
    )

plt.tight_layout()
plt.show()

### 4. Visitor Frequency Feature Check

`is_repeat_visitor` is derived from `visitor_session_count`, so these two features contain overlapping information. This table checks whether repeat behavior is associated with purchase rate.

In [ ]:
# Check relationship between visitor_session_count and is_repeat_visitor
visitor_frequency_feature_check = (
    tableau_df
    .groupby('is_repeat_visitor')
    .agg(
        sessions=('session_key', 'count'),
        avg_visitor_session_count=('visitor_session_count', 'mean'),
        median_visitor_session_count=('visitor_session_count', 'median'),
        min_visitor_session_count=('visitor_session_count', 'min'),
        max_visitor_session_count=('visitor_session_count', 'max'),
        purchase_rate=('purchased', 'mean')
    )
    .reset_index()
)

visitor_frequency_feature_check['visitor_type'] = visitor_frequency_feature_check['is_repeat_visitor'].map({
    0: 'One-time visitor',
    1: 'Repeat visitor'
})

visitor_frequency_feature_check_display = visitor_frequency_feature_check[[
    'visitor_type',
    'sessions',
    'avg_visitor_session_count',
    'median_visitor_session_count',
    'min_visitor_session_count',
    'max_visitor_session_count',
    'purchase_rate'
]].copy()

visitor_frequency_feature_check_display['purchase_rate'] = (
    visitor_frequency_feature_check_display['purchase_rate'].map(lambda x: f'{x:.2%}')
)

visitor_frequency_feature_check_display

## Quick Feature Checks

These checks confirm that the generated reporting features are sensible before exporting the dataset for Tableau.

In [ ]:
# Purchase rate by repeat visitor flag
repeat_purchase_summary = (
    tableau_df
    .groupby('is_repeat_visitor')
    .agg(
        sessions=('session_key', 'count'),
        purchase_sessions=('purchased', 'sum'),
        purchase_rate=('purchased', 'mean')
    )
    .reset_index()
)

repeat_purchase_summary['visitor_type'] = repeat_purchase_summary['is_repeat_visitor'].map({
    0: 'One-time visitor',
    1: 'Repeat visitor'
})
repeat_purchase_summary['purchase_rate'] = repeat_purchase_summary['purchase_rate'].map(lambda x: f'{x:.2%}')

repeat_purchase_summary[['visitor_type', 'sessions', 'purchase_sessions', 'purchase_rate']]

In [ ]:
# Purchase rate by hour of day
purchase_by_hour = (
    tableau_df
    .groupby('session_start_hour')
    .agg(
        sessions=('session_key', 'count'),
        purchase_rate=('purchased', 'mean')
    )
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(
    purchase_by_hour['session_start_hour'],
    purchase_by_hour['purchase_rate'],
    marker='o',
    color='#4C78A8'
)

ax.set_title('Purchase Rate by Hour of Day')
ax.set_xlabel('Session start hour')
ax.set_ylabel('Purchase rate')
ax.set_xticks(range(0, 24))
ax.yaxis.set_major_formatter(lambda x, pos: f'{x:.1%}')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Purchase rate by day of week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

purchase_by_day = (
    tableau_df
    .groupby('session_start_day_name')
    .agg(
        sessions=('session_key', 'count'),
        purchase_rate=('purchased', 'mean')
    )
    .reindex(day_order)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(
    purchase_by_day['session_start_day_name'],
    purchase_by_day['purchase_rate'],
    color='#72B7B2'
)

ax.set_title('Purchase Rate by Day of Week')
ax.set_xlabel('Day of week')
ax.set_ylabel('Purchase rate')
ax.yaxis.set_major_formatter(lambda x, pos: f'{x:.1%}')
plt.xticks(rotation=30, ha='right')

for bar, value in zip(bars, purchase_by_day['purchase_rate']):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f'{value:.2%}',
        ha='center',
        va='bottom',
        fontsize=9
    )

plt.tight_layout()
plt.show()

### 7. Purchase Activity: Weekday vs Weekend

This chart checks whether purchase rate differs between weekday and weekend sessions.

In [ ]:
# Purchase activity across weekend vs weekday
purchase_by_weekend = (
    tableau_df
    .groupby('is_weekend')
    .agg(
        sessions=('session_key', 'count'),
        purchase_rate=('purchased', 'mean')
    )
    .reset_index()
)

purchase_by_weekend['day_type'] = purchase_by_weekend['is_weekend'].map({
    0: 'Weekday',
    1: 'Weekend'
})

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(
    purchase_by_weekend['day_type'],
    purchase_by_weekend['purchase_rate'],
    color=['#4C78A8', '#F58518']
)

ax.set_title('Purchase Rate: Weekday vs Weekend')
ax.set_xlabel('Day type')
ax.set_ylabel('Purchase rate')
ax.yaxis.set_major_formatter(lambda x, pos: f'{x:.1%}')

for bar, value in zip(bars, purchase_by_weekend['purchase_rate']):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f'{value:.2%}',
        ha='center',
        va='bottom',
        fontsize=10,
        fontweight='bold'
    )

plt.tight_layout()
plt.show()

## Save Tableau-Ready Dataset

This file is the recommended input for Tableau dashboarding and can also be used by the baseline Logistic Regression notebook.

In [ ]:
# Save Tableau-ready dataset
tableau_output_path = '../Dataset/Processed Dataset/tableau_session_funnel_data.csv'
tableau_df.to_csv(tableau_output_path, index=False)

print(f'Tableau-ready dataset saved to: {tableau_output_path}')
print(f'Rows: {len(tableau_df):,}')
print(f'Columns: {len(tableau_df.columns):,}')
print('Columns:')
print(list(tableau_df.columns))